## openWakeWord training — **voice-only** (user's language/accent)

This notebook trains a wake word **on your voice alone**. No synthetic English
positives are kept: the target pronunciation comes 100% from your recordings, so
the French "a" (or any language/accent) is respected.

**Why ~150 recordings are enough:** augmentation (room impulse responses +
background noise) turns each clip into thousands of acoustic variants. The
**negatives** (audioset, fma, ACAV features) prevent false triggers — we keep
them as-is.

**Before you start (locally):**
1. `uv run python training/record_samples.py 150` — record with YOUR mic, vary tone/distance/volume.
2. Zip the `training/training_data/<wake_word>` folder into `samples.zip`.
3. Recommended runtime: **T4 GPU**. Run the cells in order.


***NOTE:*** if Colab says the *runtime must restart* (packages already
imported), restart and rerun from the top.

Set `target_word` below **identical** to `wake_word` in `configs/ai_config.toml`.


In [ ]:
# 1. Set up piper-sample-generator (used to generate the adversarial NEGATIVES + structure)
import os, sys
if not os.path.exists("./piper-sample-generator"):
    !git clone https://github.com/rhasspy/piper-sample-generator
    !wget -O piper-sample-generator/models/en_US-libritts_r-medium.pt 'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'
    !cd piper-sample-generator && git checkout 213d4d5
    !pip install piper-tts piper-phonemize-cross
    !pip install webrtcvad
    !pip install torch==2.5.0 torchvision==0.20.0 torchaudio==2.5.0 --index-url https://download.pytorch.org/whl/cu121

target_word = 'naka'  # @param {type:"string"}   <-- must match wake_word in ai_config.toml

if "piper-sample-generator/" not in sys.path:
    sys.path.append("piper-sample-generator/")
print("target_word =", target_word)


In [ ]:
# 2. Data: RIR (room acoustics), noise/music, precomputed features (~15 min)
import locale
locale.getpreferredencoding = lambda do_setlocale=True: "UTF-8"

!git clone https://github.com/dscripka/openwakeword
!pip install -e ./openwakeword --no-deps
!pip install mutagen==1.47.0 torchinfo==1.8.0 torchmetrics==1.2.0 speechbrain==0.5.14
!pip install audiomentations==0.33.0 torch-audiomentations==0.11.0 acoustics==0.2.6
!pip install onnxruntime==1.22.1 ai_edge_litert==1.4.0 onnxsim onnx2tf onnx==1.19.1
!pip install onnx_graphsurgeon sng4onnx pronouncing==0.2.0 datasets==2.14.6 deep-phonemizer==0.0.19

os.makedirs("./openwakeword/openwakeword/resources/models", exist_ok=True)
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.onnx -O ./openwakeword/openwakeword/resources/models/embedding_model.onnx
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.tflite -O ./openwakeword/openwakeword/resources/models/embedding_model.tflite
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.onnx -O ./openwakeword/openwakeword/resources/models/melspectrogram.onnx
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.tflite -O ./openwakeword/openwakeword/resources/models/melspectrogram.tflite

import numpy as np, torch, uuid, yaml, datasets, scipy
from pathlib import Path
from tqdm import tqdm

output_dir = "./mit_rirs"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
    !git lfs install
    !git clone https://huggingface.co/datasets/davidscripka/MIT_environmental_impulse_responses
    rir_dataset = datasets.Dataset.from_dict({"audio": [str(i) for i in Path("./MIT_environmental_impulse_responses/16khz").glob("*.wav")]}).cast_column("audio", datasets.Audio())
    for row in tqdm(rir_dataset):
        name = row['audio']['path'].split('/')[-1]
        scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))

if not os.path.exists("audioset"):
    os.mkdir("audioset")
    fname = "bal_train09.tar"; out_dir = f"audioset/{fname}"
    !wget -O {out_dir} https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/bal_train09.tar
    !cd audioset && tar -xvf bal_train09.tar
    os.makedirs("./audioset_16k", exist_ok=True)
    ds = datasets.Dataset.from_dict({"audio": [str(i) for i in Path("audioset/audio").glob("**/*.flac")]}).cast_column("audio", datasets.Audio(sampling_rate=16000))
    for row in tqdm(ds):
        name = row['audio']['path'].split('/')[-1].replace(".flac", ".wav")
        scipy.io.wavfile.write(os.path.join("./audioset_16k", name), 16000, (row['audio']['array']*32767).astype(np.int16))

output_dir = "./fma"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
    fma = iter(datasets.load_dataset("rudraml/fma", name="small", split="train", streaming=True).cast_column("audio", datasets.Audio(sampling_rate=16000)))
    for i in tqdm(range(1*3600//30)):
        row = next(fma)
        name = row['audio']['path'].split('/')[-1].replace(".mp3", ".wav")
        scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))

# Precomputed openWakeWord features. Download AND verify: a truncated .npy causes
# "mmap length is greater than file size" at train time. The plain exists-check
# would keep a half-downloaded file forever — so we validate and re-fetch if bad.
_FEAT_BASE = "https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/"
for _f in ["openwakeword_features_ACAV100M_2000_hrs_16bit.npy", "validation_set_features.npy"]:
    _ok = os.path.exists(_f)
    if _ok:
        try:
            np.load(_f, mmap_mode="r")
        except Exception:
            _ok = False
            os.remove(_f)
    if not _ok:
        !wget -q --show-progress -O {_f} {_FEAT_BASE + _f}
    np.load(_f, mmap_mode="r")   # raises if still truncated → stop here, not at train time
    print("feature file OK:", _f, os.path.getsize(_f), "bytes")


In [ ]:
# 3a. Config + clip generation (adversarial negatives + structure).
# number_of_examples stays small: these synthetic English positives are
# DISCARDED in step 3b. We only keep the negatives and the folder structure.
number_of_examples       = 200    # @param {type:"slider", min:100, max:2000, step:50}
number_of_training_steps = 10000  # @param {type:"slider", min:0, max:50000, step:100}
false_activation_penalty = 1500   # @param {type:"slider", min:100, max:5000, step:50}

import sys, yaml
config = yaml.load(open("openwakeword/examples/custom_model.yml").read(), yaml.Loader)
config["target_phrase"]   = [target_word]
config["model_name"]      = target_word.replace(" ", "_")
config["n_samples"]       = number_of_examples
config["n_samples_val"]   = max(500, number_of_examples // 10)
config["steps"]           = number_of_training_steps
config["target_accuracy"] = 0.5
config["target_recall"]   = 0.25
config["output_dir"]      = "./my_custom_model"
config["max_negative_weight"]               = false_activation_penalty
config["background_paths"]                  = ['./audioset_16k', './fma']
config["false_positive_validation_data_path"] = "validation_set_features.npy"
config["feature_data_files"]                = {"ACAV100M_sample": "openwakeword_features_ACAV100M_2000_hrs_16bit.npy"}
with open('my_model.yaml', 'w') as f:
    yaml.dump(config, f)

!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --generate_clips
print("Clips generated (synthetic positives will be replaced in step 3b).")


In [ ]:
# 3b. VOICE-ONLY: DISCARD the synthetic English positives and use ONLY your
#     recordings. Upload samples.zip (your folder of wavs).
import os, glob, random, shutil, zipfile
from google.colab import files

DUPLICATE = 3   # augmentation already multiplies; a little duplication = more augmented variants

uploaded = files.upload()                       # pick samples.zip
zip_name = next(iter(uploaded))
shutil.rmtree("my_recordings", ignore_errors=True)
with zipfile.ZipFile(zip_name) as z:
    z.extractall("my_recordings")
wavs = sorted(glob.glob("my_recordings/**/*.wav", recursive=True))
assert wavs, "No .wav found in the zip!"

base      = os.path.join(config["output_dir"], config["model_name"])
pos_train = os.path.join(base, "positive_train")
pos_test  = os.path.join(base, "positive_test")

# >>> core of voice-only mode: wipe the synthetic positives <<<
for d in (pos_train, pos_test):
    shutil.rmtree(d, ignore_errors=True)
    os.makedirs(d, exist_ok=True)

random.seed(0)
sh = wavs[:]; random.shuffle(sh)
test_src = set(sh[:max(1, len(sh) // 10)])      # ~10% held out for validation
k = 0
for w in wavs:
    dst = pos_test if w in test_src else pos_train
    for _ in range(DUPLICATE):
        shutil.copy(w, os.path.join(dst, f"user_{k:05d}.wav")); k += 1
print(f"VOICE-ONLY: {len(wavs)} recordings x{DUPLICATE} -> {k} positives (0 synthetic).")


In [ ]:
# 3c. Augmentation (RIR + noise) -> training -> export -> download
import sys, os
!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --augment_clips
!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --train_model

# Naka loads the .onnx — that's the only required artifact.
onnx_path = f"my_custom_model/{config['model_name']}.onnx"
assert os.path.exists(onnx_path), (
    "Training produced no .onnx. Check the log above — the usual cause is a "
    "truncated feature .npy ('mmap length is greater than file size'): re-run "
    "the data cell, it now verifies and re-downloads bad files."
)
print("model OK:", onnx_path, os.path.getsize(onnx_path), "bytes")

# TFLite is OPTIONAL (Naka doesn't use it). onnx2tf can conflict on Colab
# ('InterpreterWrapper already registered') — that's harmless, we just skip it.
try:
    name1 = f"my_custom_model/{config['model_name']}_float32.tflite"
    name2 = f"my_custom_model/{config['model_name']}.tflite"
    !onnx2tf -i {onnx_path} -o my_custom_model/ -kat onnx____Flatten_0
    if os.path.exists(name1):
        os.replace(name1, name2)
except Exception as e:
    print("TFLite export skipped:", e)

# Drop the .onnx into the repo: models/wakeword/<wake_word>.onnx
from google.colab import files
files.download(onnx_path)
tflite_path = f"my_custom_model/{config['model_name']}.tflite"
if os.path.exists(tflite_path):
    files.download(tflite_path)
